# Notebook 1 — Data Exploration & Pipeline

This notebook covers:
1. Dataset download links & folder layout
2. Class distribution visualisation (EuroSAT & UC Merced)
3. Sample tile display (5 random tiles per class)
4. Spatial block split documentation
5. Data augmentation preview

In [ ]:
import sys
sys.path.insert(0, '../src')  # allow imports from src/

from pathlib import Path
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from PIL import Image

from landuse.data import load_image_folder, split_dataset, build_transforms

# --- Paths (edit these to match your local layout) ---
EUROSAT_DIR   = Path('../data/eurosat')
UC_MERCED_DIR = Path('../data/uc_merced')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('EuroSAT exists  :', EUROSAT_DIR.exists())
print('UC Merced exists:', UC_MERCED_DIR.exists())

## 1. Dataset Download

| Dataset | Source | Size |
|---|---|---|
| EuroSAT RGB | [zenodo.org/record/7711810](https://zenodo.org/record/7711810) | ~90 MB |
| UC Merced Land Use | [vision.ucmerced.edu](http://vision.ucmerced.edu/datasets/landuse.html) | ~340 MB |

Download and extract so the folder layout matches `README.md`.

In [ ]:
# --- Class distribution: EuroSAT ---
dataset = load_image_folder(EUROSAT_DIR, train=False)
counts = pd.Series([dataset.classes[y] for _, y in dataset.samples]).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='viridis')
ax.set_title('EuroSAT — Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Land-use class')
ax.set_ylabel('Number of images')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig('../runs/data_profile/eurosat_class_dist.png', dpi=150)
plt.show()
print(counts)

In [ ]:
# --- Class distribution: UC Merced ---
uc_dataset = load_image_folder(UC_MERCED_DIR, train=False)
uc_counts = pd.Series([uc_dataset.classes[y] for _, y in uc_dataset.samples]).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
sns.barplot(x=uc_counts.index, y=uc_counts.values, ax=ax, palette='plasma')
ax.set_title('UC Merced — Class Distribution (21 classes)', fontsize=14, fontweight='bold')
ax.set_xlabel('Land-use class')
ax.set_ylabel('Number of images')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../runs/data_profile/uc_merced_class_dist.png', dpi=150)
plt.show()
print(uc_counts)

In [ ]:
# --- Sample tiles: 5 random images per EuroSAT class ---
from collections import defaultdict

by_class = defaultdict(list)
for path, label in dataset.samples:
    by_class[label].append(path)

n_classes = len(dataset.classes)
samples_per_class = 5
fig, axes = plt.subplots(n_classes, samples_per_class,
                          figsize=(samples_per_class * 2, n_classes * 2))
fig.suptitle('EuroSAT — 5 Random Tiles per Class', fontsize=14, fontweight='bold')

for row, (label, cls_name) in enumerate(enumerate(dataset.classes)):
    paths = random.sample(by_class[label], min(samples_per_class, len(by_class[label])))
    for col, p in enumerate(paths):
        img = Image.open(p).convert('RGB')
        ax = axes[row][col]
        ax.imshow(img)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls_name, rotation=0, labelpad=60, va='center', fontsize=8)

plt.tight_layout(rect=[0.08, 0, 1, 0.97])
plt.savefig('../runs/data_profile/eurosat_sample_tiles.png', dpi=120)
plt.show()

In [ ]:
# --- Spatial block split documentation ---
import hashlib

def block_id(path: str, blocks: int = 20) -> int:
    digest = hashlib.sha1(Path(path).stem.encode('utf-8')).hexdigest()
    return int(digest[:8], 16) % blocks

splits = split_dataset(dataset, split='block')
print(f'Block split sizes — Train: {len(splits.train):,}  Val: {len(splits.val):,}  Test: {len(splits.test):,}')

# Show which blocks ended up in each split
train_paths = [dataset.samples[i][0] for i in splits.train.indices]
test_paths  = [dataset.samples[i][0] for i in splits.test.indices]

train_blocks = sorted(set(block_id(p) for p in train_paths))
test_blocks  = sorted(set(block_id(p) for p in test_paths))
print(f'Train blocks : {train_blocks}')
print(f'Test blocks  : {test_blocks}')
print('\nNo block appears in both train and test — spatial leakage is prevented.')

In [ ]:
# --- Augmentation preview ---
train_tfm  = build_transforms(train=True)
infer_tfm  = build_transforms(train=False)

# Pick one example image
example_path = dataset.samples[0][0]
orig = Image.open(example_path).convert('RGB')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def denorm(t):
    t = t.permute(1, 2, 0).numpy()
    t = t * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(t, 0, 1)

n_aug = 6
fig, axes = plt.subplots(1, n_aug + 1, figsize=(14, 3))
axes[0].imshow(orig)
axes[0].set_title('Original', fontsize=9)
axes[0].axis('off')

for i in range(n_aug):
    aug = train_tfm(orig)
    axes[i + 1].imshow(denorm(aug))
    axes[i + 1].set_title(f'Aug {i+1}', fontsize=9)
    axes[i + 1].axis('off')

fig.suptitle('Data Augmentation Preview (RandomFlip + RandomRotation)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../runs/data_profile/augmentation_preview.png', dpi=120)
plt.show()